In [ ]:
import os, sys, time, argparse
import matplotlib.pyplot as plt
import numpy as np
import json
import ray

def load_swarm_data(filename):
    with open(filename) as json_file:
        swarm_data = json.load(json_file)
        
    # Grab the total number of integrations
    n_data = len(swarm_data)

    # 'cal_solution', 'delays', 'efficiencies', 'inputs', 'int_length', 'int_time', 'phases'
    if 'phases' in swarm_data[0].keys():
        # Phases is the "old" keyword, where DSB phasing wasn't used.
        input_count = np.median([len(swarm_data[idx]['inputs']) for idx in range(n_data)])
        use_data = [
            (len(swarm_data[idx]['inputs']) == input_count)
            and (len(swarm_data[idx]['phases']) == input_count)
            and (len(swarm_data[idx]['cal_solution'][2]) == input_count)
            and (len(swarm_data[idx]['efficiencies']) == 8)
            for idx in range(n_data)
        ]
        

        swarm_data = [swarm_data[idx] for idx in range(n_data) if use_data[idx]]
        n_data = len(swarm_data)

        n_inputs = len(np.unique(np.array([data['inputs'] for data in swarm_data])[:, :, 0]))
        
        # These are the implemented phase values recorded in SWARM
        phase_online = np.array([swarm_data[idx]['phases'] for idx in range(n_data)])
        # These are the derived phase offsets post-correlation
        phase_solns = np.array([swarm_data[idx]['cal_solution'][2] for idx in range(n_data)])
        efficiencies  = np.array([swarm_data[idx]['efficiencies'] for idx in range(n_data)])
    else:
        input_count = np.median([[len(data['inputs_lsb']), len(data['inputs_usb'])] for data in swarm_data])

        use_data = [
            (len(swarm_data[idx]['inputs_lsb']) == input_count)
            and (len(swarm_data[idx]['phases_lsb']) == input_count)
            and (len(swarm_data[idx]['cal_solution_lsb'][2]) == input_count)
            and (len(swarm_data[idx]['inputs_usb']) == input_count)
            and (len(swarm_data[idx]['phases_usb']) == input_count)
            and (len(swarm_data[idx]['cal_solution_usb'][2]) == input_count)
            for idx in range(n_data)
        ]

        swarm_data = [swarm_data[idx] for idx in range(n_data) if use_data[idx]]
        n_data = len(swarm_data)

        n_inputs = len(np.unique(np.array(
            [[data['inputs_lsb'], data['inputs_usb']] for data in swarm_data]
        )[:, :, :, 0]))

        # We're gonna be doing a lot of diff operations, which means in some cases we'll want
        # to pad some arrays with zeros. Construct some arrays now for the sake of convenience
        # These are the implemented phase values recorded in SWARM
        phase_online = np.concatenate(
            (
                np.array([data['phases_lsb'] for data in swarm_data]),
                np.array([data['phases_usb'] for data in swarm_data]),
            ),
            axis=1,
        )

        # These are the derived offsets/error terms for each antenna, given the implemented values
        phase_solns = np.concatenate(
            (
                np.array([data['cal_solution_lsb'][2] for data in swarm_data]),
                np.array([data['cal_solution_usb'][2] for data in swarm_data])
            ),
            axis=1,
        )

        efficiencies = np.concatenate(
            (
                np.array([data['efficiencies_lsb'] for data in swarm_data]),
                np.array([data['efficiencies_usb'] for data in swarm_data])
            ),
            axis = 1,
        )

    # Let's calculate the "true" phase -- that is, assume that the solutions are perfect, and
    # use that to figure out what the antenna phase should _actually_ have been at time of obs.
    # There's kind of a funny padding operation that's needed here because of the order values
    # in the JSON file are recorded (soln's derived -> values implemented -> values recorded).
    # Add the two to get the "true" value at the time
    true_phases = phase_online[:-1] + phase_solns[1:]

        #true_phases = phases_usb[:-1] + cal_solution_usb[1:]
        #prog_vals = phases_usb

    # Convert times from UNIX -> fractional UTC hours
    time_stamps = (np.array([data['int_time'] for data in swarm_data]) % 86400) / 3600.0
    
    return (true_phases, n_inputs, time_stamps, phase_online, efficiencies)



In [ ]:
### Data file names
data = ('../phasing_data/vlbi_cal.095-2017.json',
        '../phasing_data/vlbi_cal.096-2017.json',
        '../phasing_data/vlbi_cal.097-2017.json',
        '../phasing_data/vlbi_cal.100-2017.json',
        '../phasing_data/vlbi_cal.101-2017.json',
        '../phasing_data/vlbi_cal.195-2017.json',
        '../phasing_data/vlbi_cal.349-2017.json')

for idx in range(len(data)):
    swarm_data = load_swarm_data(data[idx])

In [ ]:
with open(data[0]) as json_file:
    swarm_data = json.load(json_file)


In [83]:
marker = 0
while marker < len(swarm_data):
    if len(swarm_data[marker]['efficiencies']) != 8:
        _ = swarm_data.pop(marker)
    else:
        marker += 1

In [85]:
len(swarm_data[0]['efficiencies'])

8